# S&P 500 Daily Returns: Data Validation and Summary Statistics

FIM 500 (NC State MFM) · *Forecasting S&P 500 Conditional Volatility with a GJR-GARCH Model*

This notebook builds the modeling dataset, checks it for data errors, and produces every number on the Data slide.

- **Data.** Daily ^GSPC closes from Yahoo Finance, 2006-01-03 to 2026-09-18, converted to daily log returns. This is the same source and window as `S&P500_datacleaned_Parin Sharda.ipynb`.
- **Decision.** All 5,209 returns are used. Validation finds no data errors. The four days removed by the original 8σ screen are real market moves (§2.5), and removing them distorts the volatility model (§2.6). The 8σ-cleaned series is reported alongside as a robustness check.

**Contents:** 1 Data · 2 Validation · 3 Summary statistics

## Setup

In [1]:
import warnings

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import yfinance as yf
from arch import arch_model
from IPython.display import display
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.width", 140)

In [2]:
TICKER = "^GSPC"
START_DATE, END_DATE = "2006-01-01", "2026-09-20"  # END_DATE is exclusive in yfinance
TRADING_DAYS = 252
LB_LAGS = [10, 20]

OUTLIER_Z = 8           # original screen from Parin's notebook, kept for the robustness column
EWMA_LAMBDA = 0.94      # RiskMetrics decay for the conditional z-score
SPIKE_MIN = 0.03        # spike-and-revert check: |log return| above 3% ...
REVERT_SHARE = 0.90     # ... with more than 90% of it undone the next day
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500"  # official S&P DJI closes, last 10 years

In [3]:
def log_returns(close: pd.Series) -> pd.Series:
    """Daily log returns, today vs. the previous trading day."""
    return np.log(close / close.shift(1)).dropna()


def summarize(r: pd.Series) -> dict:
    """Every Data-slide statistic for a daily log-return series in decimal units."""
    jb = stats.jarque_bera(r)
    with warnings.catch_warnings():
        # statsmodels 0.15 warns that adfuller's tuple return changes in 0.16
        warnings.simplefilter("ignore", FutureWarning)
        adf_stat, adf_p, adf_lags, _, adf_crit = adfuller(r, autolag="AIC")[:5]
    lb = {"returns": acorr_ljungbox(r, lags=LB_LAGS), "squared returns": acorr_ljungbox(r**2, lags=LB_LAGS)}
    return {
        "n": len(r),
        "dates": (r.index[0].date(), r.index[-1].date()),
        "mean": r.mean(),
        "std": r.std(),
        "min": (r.min(), r.idxmin().date()),
        "max": (r.max(), r.idxmax().date()),
        "skew": r.skew(),
        "ex_kurt": r.kurt(),  # pandas kurt() is excess (Fisher) kurtosis
        "jb": (jb.statistic, jb.pvalue),
        "adf": (adf_stat, adf_p, adf_lags, adf_crit["1%"]),
        "lb": {(name, lag): (t.loc[lag, "lb_stat"], t.loc[lag, "lb_pvalue"]) for name, t in lb.items() for lag in LB_LAGS},
    }


def fmt_p(p: float) -> str:
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


def slide_column(s: dict) -> dict:
    """Format summarize() output as slide-ready strings (returns in %)."""
    (min_r, min_d), (max_r, max_d) = s["min"], s["max"]
    adf_stat, adf_p, adf_lags, adf_crit_1 = s["adf"]
    col = {
        "Return dates": f"{s['dates'][0]} to {s['dates'][1]}",
        "Observations (daily log returns)": f"{s['n']:,}",
        "Mean daily log return (%)": f"{s['mean'] * 100:.4f}",
        "Daily std (%)": f"{s['std'] * 100:.3f}",
        "Annualized vol (%)": f"{s['std'] * np.sqrt(TRADING_DAYS) * 100:.2f}",
        "Min daily return (%)": f"{min_r * 100:.2f} ({min_d})",
        "Max daily return (%)": f"{max_r * 100:.2f} ({max_d})",
        "Skewness": f"{s['skew']:.3f}",
        "Excess kurtosis": f"{s['ex_kurt']:.2f}",
        "Jarque-Bera": f"{s['jb'][0]:,.0f} ({fmt_p(s['jb'][1])})",
        "ADF stat (AIC lag selection)": f"{adf_stat:.2f} ({fmt_p(adf_p)})",
        "ADF lags used / 1% critical value": f"{adf_lags} / {adf_crit_1:.3f}",
    }
    for (name, lag), (q, p) in s["lb"].items():
        col[f"Ljung-Box Q({lag}), {name}"] = f"{q:,.1f} ({fmt_p(p)})"
    return col

## 1. Data

In [4]:
# yfinance auto-adjusts by default; for an index, Close and Adj Close are identical
prices = yf.download(TICKER, START_DATE, END_DATE, multi_level_index=False, progress=False)
close = prices["Close"]
r = log_returns(close)

assert (len(close), len(r)) == (5210, 5209), "Yahoo data differs from the 2026-10-13 run"
print(f"Prices : {close.index[0].date()} to {close.index[-1].date()}  ({len(close):,} closes)")
print(f"Returns: {r.index[0].date()} to {r.index[-1].date()}  ({len(r):,} daily log returns)")

Prices : 2006-01-03 to 2026-09-18  (5,210 closes)
Returns: 2006-01-04 to 2026-09-18  (5,209 daily log returns)


## 2. Data validation

Each check below looks for a specific kind of data error, not just for large moves.

### 2.1 Trading calendar
Yahoo's dates should match NYSE trading days exactly, including special closures such as Hurricane Sandy in 2012 and national days of mourning.

In [5]:
nyse_days = mcal.get_calendar("NYSE").valid_days(close.index[0], close.index[-1]).tz_localize(None)
print(f"NYSE trading days: {len(nyse_days):,}   Yahoo days: {len(close):,}")
print(f"Missing from Yahoo: {list(nyse_days.difference(close.index).date)}")
print(f"Not NYSE trading days: {list(close.index.difference(nyse_days).date)}")

NYSE trading days: 5,210   Yahoo days: 5,210
Missing from Yahoo: []
Not NYSE trading days: []


### 2.2 Integrity
A stale price shows up as an unchanged close. If the index still traded through a normal intraday range that day, the unchanged close is real.

In [6]:
unchanged = r.index[r == 0]
display(pd.Series({
    "Missing values": close.isna().sum(),
    "Duplicate dates": close.index.duplicated().sum(),
    "Non-positive closes": (close <= 0).sum(),
    "Unchanged closes": len(unchanged),
}).to_frame("Count"))
prices.loc[unchanged, ["Open", "High", "Low", "Close"]].assign(Range=lambda d: d["High"] - d["Low"]).round(2)

,Count
Missing values,0
Duplicate dates,0
Non-positive closes,0
Unchanged closes,2


,Open,High,Low,Close,Range
Date,,,,,
2008-01-03,1447.55,1456.80,1443.73,1447.16,13.07
2017-01-10,2269.72,2279.27,2265.27,2268.90,14.00


Both unchanged closes had intraday ranges of 13 to 14 points, so they are genuine.

### 2.3 Spike-and-revert
A bad tick usually looks like a large move that is undone the next day, typically in an otherwise calm market.

In [7]:
next_r = r.shift(-1)
spikes = (r.abs() > SPIKE_MIN) & ((r + next_r).abs() < (1 - REVERT_SHARE) * r.abs())
pd.DataFrame({"Return (%)": r[spikes] * 100, "Next day (%)": next_r[spikes] * 100}).round(2)

,Return (%),Next day (%)
Date,,
2008-09-19,3.95,-3.90
2009-01-28,3.30,-3.37
2011-08-09,4.63,-4.52
2011-08-10,-4.52,4.53
2020-03-10,4.82,-5.01
2020-03-17,5.82,-5.32
2020-03-27,-3.43,3.30


Every candidate falls inside a known crisis: Sept 2008 (TARP), Jan 2009, the August 2011 US downgrade, and March 2020. Large swings in both directions are normal in those periods, and none of these is an isolated spike in a calm market. The March 2020 days also match the official closes in §2.4.

### 2.4 Cross-check against official closes (FRED)
FRED's `SP500` series comes from S&P Dow Jones Indices and covers the most recent 10 years.

In [8]:
fred = pd.read_csv(FRED_URL, index_col="observation_date", parse_dates=True)["SP500"]
fred = pd.to_numeric(fred, errors="coerce").dropna()  # holidays are blank

both = pd.concat({"Yahoo": close, "Official": fred}, axis=1, join="inner")
both["Diff (pts)"] = both["Yahoo"] - both["Official"]
mismatch = both[both["Diff (pts)"].abs() > 0.01]
print(f"Overlap {both.index[0].date()} to {both.index[-1].date()}: {len(both):,} days, "
      f"{len(mismatch)} closes differ by more than 0.01 pts")
mismatch.round(2)

Overlap 2016-10-13 to 2026-09-18: 2,496 days, 8 closes differ by more than 0.01 pts


,Yahoo,Official,Diff (pts)
2018-11-29,2737.80,2737.76,0.04
2019-08-12,2882.70,2883.75,-1.05
2020-05-11,2930.19,2930.32,-0.13
2020-10-02,3348.42,3348.44,-0.02
2020-10-05,3408.60,3408.63,-0.03
2020-10-06,3360.97,3360.95,0.02
2020-10-07,3419.44,3419.45,-0.01
2021-08-11,4442.41,4447.70,-5.29


Only two of these differences are more than rounding: 2019-08-12 and 2021-08-11. To check whether they matter, the cell below replaces Yahoo's closes with the official ones wherever they overlap and lists every slide value that changes:

In [9]:
close_official = fred.reindex(close.index).combine_first(close)
yahoo_col = slide_column(summarize(r))
official_col = slide_column(summarize(log_returns(close_official)))
pd.DataFrame(
    {k: {"Yahoo": yahoo_col[k], "Official closes": official_col[k]} for k in yahoo_col if yahoo_col[k] != official_col[k]}
).T

,Yahoo,Official closes
Jarque-Bera,"36,523 (p < 0.001)","36,527 (p < 0.001)"
"Ljung-Box Q(20), squared returns","7,100.0 (p < 0.001)","7,100.1 (p < 0.001)"


The differences are immaterial, so the Yahoo series is used as is.

### 2.5 The original 8σ screen

Parin's notebook dropped returns with |z| > 8, where z uses the full-sample mean and std. That measures each day against average volatility over 20 years. GJR-GARCH assumes volatility changes over time, so the relevant comparison is the volatility *at the time*. The conditional z below divides each return by an EWMA volatility estimate (λ = 0.94) that uses only information up to the previous day.

In [10]:
z = (r - r.mean()) / r.std()
flagged = r.index[z.abs() > OUTLIER_Z]
r_clean = r[z.abs() < OUTLIER_Z]  # Parin's keep rule, for the robustness column
assert (len(flagged), len(r_clean)) == (4, 5205), "Does not reproduce Parin's screen"

ewma_vol = np.sqrt((r**2).ewm(alpha=1 - EWMA_LAMBDA, adjust=False).mean().shift(1))
pd.DataFrame({
    "Prev close": close.shift(1)[flagged],
    "Close": close[flagged],
    "Next close": close.shift(-1)[flagged],
    "Return (%)": r[flagged] * 100,
    "z (full sample)": z[flagged],
    "Vol going in (%)": ewma_vol[flagged] * 100,
    "z (conditional)": r[flagged] / ewma_vol[flagged],
}).round(2)

,Prev close,Close,Next close,Return (%),z (full sample),Vol going in (%),z (conditional)
Date,,,,,,,
2008-10-13,899.22,1003.35,998.01,10.96,8.96,3.72,2.94
2008-10-28,848.92,940.51,930.09,10.25,8.38,4.44,2.31
2020-03-12,2741.38,2480.64,2711.02,-9.99,-8.23,3.21,-3.12
2020-03-16,2711.02,2386.13,2529.19,-12.77,-10.51,4.41,-2.89


All four are documented market events, and none reverts the next day. Each is only a 2 to 3σ move against the volatility at the time:

- 2008-10-13: rally after the G7 and European bank-rescue announcements
- 2008-10-28: rally during the October 2008 crisis
- 2020-03-12: COVID crash; market-wide circuit breaker
- 2020-03-16: COVID crash; circuit breaker after the Fed's emergency cut to zero

### 2.6 Effect of the 8σ screen on a GJR-GARCH fit
GJR-GARCH(1,1,1) with a constant mean and skewed-t innovations, fit to returns in percent (the scaling `arch` recommends). This is a robustness check, not the final model.

In [11]:
def fit_gjr(returns: pd.Series):
    return arch_model(returns * 100, mean="Constant", vol="GARCH", p=1, o=1, q=1, dist="skewt").fit(disp="off")


def gjr_params(res) -> pd.Series:
    p = res.params
    persistence = p["alpha[1]"] + p["gamma[1]"] / 2 + p["beta[1]"]
    return pd.Series({
        "omega": p["omega"],
        "alpha": p["alpha[1]"],
        "gamma": p["gamma[1]"],
        "gamma t-stat": res.tvalues["gamma[1]"],
        "beta": p["beta[1]"],
        "persistence (alpha + gamma/2 + beta)": persistence,
        "long-run annualized vol (%)": np.sqrt(p["omega"] / (1 - persistence) * TRADING_DAYS),
        "skewed-t dof (eta)": p["eta"],
        "skewed-t skew (lambda)": p["lambda"],
    })


fits = {"Full sample": fit_gjr(r), "8σ-cleaned": fit_gjr(r_clean)}
display(pd.DataFrame({name: gjr_params(res) for name, res in fits.items()}).round(4))

# Conditional vol on the trading day after each removed day: the cleaned fit never sees the shock
day_after = r.index[r.index.get_indexer(flagged) + 1]
pd.DataFrame(
    {
        **{f"Model vol, {name} (%)": res.conditional_volatility.reindex(day_after).values for name, res in fits.items()},
        "Actual |return| (%)": r[day_after].abs().values * 100,
    },
    index=pd.Index(day_after.date, name="Day after removed day"),
).round(2)

,Full sample,8σ-cleaned
omega,0.0233,0.0227
alpha,0.0000,0.0000
gamma,0.2432,0.2397
gamma t-stat,8.7561,8.5317
beta,0.8626,0.8647
persistence (alpha + gamma/2 + beta),0.9842,0.9845
long-run annualized vol (%),19.2758,19.2158
skewed-t dof (eta),6.4332,6.4596
skewed-t skew (lambda),-0.1617,-0.1605


,"Model vol, Full sample (%)","Model vol, 8σ-cleaned (%)",Actual |return| (%)
Day after removed day,,,
2008-10-14,5.15,5.53,0.53
2008-10-29,4.72,5.18,1.11
2020-03-13,6.67,4.79,8.88
2020-03-17,8.54,4.46,5.82


The fitted parameters barely change, because the skewed-t distribution already handles the fat tails. The volatility path does change. In March 2020 the cleaned fit's next-day volatility is 28% to 48% lower than the full-sample fit's, because the deleted shocks never enter the variance recursion. Removing the days gains nothing in estimation and hurts the forecasts when they matter most.

**Data decision: use all 5,209 returns.**

## 3. Summary statistics
Slide-ready values. The full sample is used for modeling. The 8σ-cleaned column is a robustness check.

In [12]:
price_dates = f"{close.index[0].date()} to {close.index[-1].date()}"
summary = pd.DataFrame({
    "Full sample (used)": {"Price dates": price_dates, "Days removed": "0", **slide_column(summarize(r))},
    "8σ-cleaned (robustness)": {"Price dates": price_dates, "Days removed": f"{len(flagged)}", **slide_column(summarize(r_clean))},
})
summary.index.name = "Statistic"
summary

,Full sample (used),8σ-cleaned (robustness)
Statistic,,
Price dates,2006-01-03 to 2026-09-18,2006-01-03 to 2026-09-18
Days removed,0,4
Return dates,2006-01-04 to 2026-09-18,2006-01-04 to 2026-09-18
Observations (daily log returns),"5,209","5,205"
Mean daily log return (%),0.0345,0.0348
Daily std (%),1.218,1.180
Annualized vol (%),19.34,18.73
Min daily return (%),-12.77 (2020-03-16),-9.47 (2008-10-15)
Max daily return (%),10.96 (2008-10-13),9.09 (2025-04-09)
